In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/v3/development_v3.csv")

# Fix tipi (obbligatorio)
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("unknown").astype(str)

# ---------- SOURCE GROUPING ----------
def normalize_source(s):
	s = s.lower()
	if "reuters" in s:
		return "reuters"
	if "bbc" in s:
		return "bbc"
	if "cnn" in s:
		return "cnn"
	if s.startswith("ap"):
		return "ap"
	if "fox" in s:
		return "fox"
	return "other"

df["source_group"] = df["source"].apply(normalize_source)


In [2]:
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, confusion_matrix


In [3]:
X = df.drop(columns=["label"])
y = df["label"]

text_cols = ["article", "title"]
num_cols  = ["title_ratio", "n_tokens", "source_support"]
cat_cols  = ["source_group"]

preprocess = ColumnTransformer(
	transformers=[
		("article", TfidfVectorizer(
			max_features=120_000,
			ngram_range=(1,2),
			min_df=3,
			max_df=0.9,
			sublinear_tf=True,
			stop_words="english"
		), "article"),

		("title", TfidfVectorizer(
			max_features=30_000,
			ngram_range=(1,2),
			min_df=2,
			max_df=0.95,
			sublinear_tf=True,
			stop_words="english"
		), "title"),

		("num", StandardScaler(), num_cols),

		("src", OneHotEncoder(handle_unknown="ignore"), cat_cols)
	],
	n_jobs=-1
)

model = Pipeline([
	("prep", preprocess),
	("clf", LogisticRegression(
		C=1.0,
		max_iter=1000,
		n_jobs=-1
	))
])


In [4]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1s = []
recalls = []
cms = []

for tr, te in skf.split(X, y):
	model.fit(X.iloc[tr], y.iloc[tr])
	yp = model.predict(X.iloc[te])

	f1s.append(f1_score(y.iloc[te], yp, average="macro"))
	recalls.append(recall_score(y.iloc[te], yp, average="macro"))
	cms.append(confusion_matrix(y.iloc[te], yp))

print("FINAL MODEL – Semantic + minimal numeric + source_group")
print("Macro F1:", np.mean(f1s))
print("Macro Recall:", np.mean(recalls))
print("Confusion Matrix:\n", np.sum(cms, axis=0))


FINAL MODEL – Semantic + minimal numeric + source_group
Macro F1: 0.6714300884046377
Macro Recall: 0.6699530063981777
Confusion Matrix:
 [[18563   789   488   832   307  2317   245]
 [  789  7947   890   320    43   473   126]
 [  811   822  8619   422    80   278   129]
 [ 1944   588   611  4658   650  1315   211]
 [  280    51    65   304  7549   316     9]
 [ 4050   905   466  1057   657  5631   287]
 [  464   137   177   174    18   272  1860]]


In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, confusion_matrix

# =====================
# LOAD DATA
# =====================
df = pd.read_csv("../data/processed/v3/development_v3.csv")

# Fix definitivi per TF-IDF
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)

X = df.drop(columns=["label"])
y = df["label"]

# =====================
# COLUMNS
# =====================
text_article = "article"
text_title   = "title"

num_cols = [
	"n_tokens",
	"title_ratio"
]

# =====================
# PREPROCESSOR
# =====================
preprocess = ColumnTransformer(
	transformers=[
		# Article – word ngrams
		("article_word", TfidfVectorizer(
			max_features=120_000,
			ngram_range=(1,2),
			min_df=3,
			max_df=0.9,
			sublinear_tf=True,
			stop_words="english"
		), text_article),

		# Title – word ngrams
		("title_word", TfidfVectorizer(
			max_features=30_000,
			ngram_range=(1,2),
			min_df=2,
			max_df=0.95,
			sublinear_tf=True,
			stop_words="english"
		), text_title),

		# 🔥 Title – CHAR ngrams (KEY TRICK)
		("title_char", TfidfVectorizer(
			analyzer="char",
			ngram_range=(3,5),
			min_df=3,
			sublinear_tf=True
		), text_title),

		# Minimal numeric
		("num", StandardScaler(), num_cols)
	],
	n_jobs=-1
)

# =====================
# MODEL
# =====================
model = Pipeline([
	("prep", preprocess),
	("clf", LogisticRegression(
		C=1.0,
		max_iter=1000,
		n_jobs=-1
	))
])

# =====================
# CV EVALUATION
# =====================
skf = StratifiedKFold(
	n_splits=5,
	shuffle=True,
	random_state=42
)

f1s = []
recalls = []
cms = []

for tr, te in skf.split(X, y):
	model.fit(X.iloc[tr], y.iloc[tr])
	yp = model.predict(X.iloc[te])

	f1s.append(f1_score(y.iloc[te], yp, average="macro"))
	recalls.append(recall_score(y.iloc[te], yp, average="macro"))
	cms.append(confusion_matrix(y.iloc[te], yp))

print("STRATEGY 2 + TITLE CHAR")
print("Macro F1:", np.mean(f1s))
print("Macro Recall:", np.mean(recalls))
print("Confusion Matrix:\n", np.sum(cms, axis=0))


STRATEGY 2 + TITLE CHAR
Macro F1: 0.6611722130260548
Macro Recall: 0.6592416875198113
Confusion Matrix:
 [[18198   820   597   881   375  2395   275]
 [  824  7757   856   331    47   642   131]
 [  992   781  8414   402    77   368   127]
 [ 2170   554   539  4715   629  1158   212]
 [  411    49    49   314  7420   322     9]
 [ 4158   899   545  1079   685  5383   304]
 [  495   143   141   169    22   256  1876]]


In [10]:
df = pd.read_csv("../data/raw/development.csv")
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("unknown").astype(str)

X = df.drop(columns=["label"])
y = df["label"]

In [11]:
from sklearn.cluster import MiniBatchKMeans

def fit_semantic_cluster(train_text, n_clusters=12):
	vec = TfidfVectorizer(
		max_features=80_000,
		ngram_range=(1,2),
		min_df=5,
		stop_words="english",
		sublinear_tf=True
	)
	X = vec.fit_transform(train_text)

	kmeans = MiniBatchKMeans(
		n_clusters=n_clusters,
		random_state=42,
		batch_size=2048
	)

	labels = kmeans.fit_predict(X)
	return vec, kmeans, labels


In [12]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1s = []
cms = []

for tr, te in skf.split(X, y):

	df_tr = df.iloc[tr].copy()
	df_te = df.iloc[te].copy()

	# ---- semantic clustering ----
	vec_c, km, tr_clusters = fit_semantic_cluster(
		df_tr["article"], n_clusters=12
	)

	df_tr["semantic_cluster"] = tr_clusters
	df_te["semantic_cluster"] = km.predict(
		vec_c.transform(df_te["article"])
	)

	# ---- model ----
	preprocess = ColumnTransformer(
		transformers=[
			("article", TfidfVectorizer(
				max_features=120_000,
				ngram_range=(1,2),
				min_df=3,
				max_df=0.9,
				sublinear_tf=True,
				stop_words="english"
			), "article"),

			("title", TfidfVectorizer(
				max_features=30_000,
				ngram_range=(1,2),
				min_df=2,
				max_df=0.95,
				sublinear_tf=True,
				stop_words="english"
			), "title"),

			("source", OneHotEncoder(handle_unknown="ignore"), ["source"]),

			("cluster", OneHotEncoder(handle_unknown="ignore"),
			 ["semantic_cluster"])
		],
		n_jobs=-1
	)

	model = Pipeline([
		("prep", preprocess),
		("clf", LogisticRegression(
			C=1.0,
			max_iter=1000,
			n_jobs=-1
		))
	])

	model.fit(df_tr, df_tr["label"])
	yp = model.predict(df_te)

	f1s.append(f1_score(df_te["label"], yp, average="macro"))
	cms.append(confusion_matrix(df_te["label"], yp))


In [15]:
print("Semantic + Title + Source + Article-Cluster")
print("Macro F1:", np.mean(f1s))
print("Confusion Matrix:\n", np.sum(cms, axis=0))


Semantic + Title + Source + Article-Cluster
Macro F1: 0.7009251405594525
Confusion Matrix:
 [[18894   643   417   777   207  2371   233]
 [  746  8327   543   350    71   439   112]
 [  694   624  9093   336    49   254   111]
 [ 1687   573   516  4971   622  1394   214]
 [  235    55    15   314  7613   338     4]
 [ 3810   633   295  1138   643  6261   273]
 [  441   154    74   213    36   280  1904]]
